# Anticipatory Energy–Carbon Stress Index — Monitoring Dashboard

**Project**: Anticipatory Energy–Carbon Stress Index (UK, 2005–2018)  
**Data range**: 2005-01-01 → 2018-12-01  
**Forecast target**: Monthly 2018 (Jan–Dec)

Three analytical baselines:
| Baseline | Description |
|----------|-------------|
| `FES_core`   | Forecasted using core-only models (gas, electricity, carbon) |
| `FES_macro`  | Forecasted using core + macro exogenous (inflation, weather, GDP) |
| `FES_actual` | Realised 2018 values — benchmark |

Run all cells in order after executing `python run_pipeline.py` from the project root.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
except ImportError:
    pass

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120})

PALETTE = {'gas': '#E67E22', 'electricity': '#2980B9', 'carbon': '#27AE60',
           'core': '#8E44AD', 'macro': '#E74C3C', 'actual': '#2C3E50'}

print('Environment ready.')

## 1 — Historical Data (2005–2018)

In [ ]:
try:
    core  = pd.read_csv('../data/raw/core_energy_carbon.csv',  parse_dates=['date'])
    macro = pd.read_csv('../data/raw/macro_controls.csv',      parse_dates=['date'])
    print(f'Core  dataset : {core.shape}  ({core.date.min().year}–{core.date.max().year})')
    print(f'Macro dataset : {macro.shape}  ({macro.date.min().year}–{macro.date.max().year})')
    print('\nAll core series are YoY growth rates (%) — conceptually consistent.')
    print(core.describe().round(3).to_string())
except FileNotFoundError:
    print('Data not generated yet. Run: python run_pipeline.py')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
series_info = [
    ('gas_growth',         'Gas Growth (YoY %)',          PALETTE['gas']),
    ('electricity_growth', 'Electricity Growth (MoM %)',  PALETTE['electricity']),
    ('carbon_growth',      'Carbon Growth (YoY %)',        PALETTE['carbon']),
]
for ax, (col, label, colour) in zip(axes, series_info):
    ax.plot(core['date'], core[col], color=colour, linewidth=1.8)
    ax.fill_between(core['date'], core[col], 0, alpha=0.12, color=colour)
    ax.axhline(0, color='#BDC3C7', linewidth=0.8)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
fig.suptitle('UK Energy–Carbon Growth Rates (2005–2018)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2 — Macro Control Variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
macro_series = [
    ('inflation_growth',   'UK CPI Inflation (MoM %)',      '#8E44AD'),
    ('weather_volatility', 'UK Weather Volatility (RollStd)','#1ABC9C'),
    ('gdp_growth',         'UK GDP Growth (MoM %)',          '#E74C3C'),
]
for ax, (col, label, colour) in zip(axes, macro_series):
    ax.plot(macro['date'], macro[col], color=colour, linewidth=1.6)
    ax.axhline(0, color='#BDC3C7', linewidth=0.8, linestyle=':')
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
fig.suptitle('Macro Control Variables — UK (2005–2018)\n'
             '(exogenous inputs only; NOT FES components)',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 3 — Model Ranking (2017 Evaluation)

In [ ]:
try:
    metrics = pd.read_csv('../outputs/tables/model_metrics_comparison.csv')
    print(f'Loaded metrics table: {metrics.shape}')
    display_cols = [c for c in ['series_name','mode','model','MAE','RMSE','SMAPE','MASE','rank_score']
                    if c in metrics.columns]
    print('\nNote: MAPE excluded from ranking (near-zero growth rates cause explosion).')
    print('Ranking uses: MAE, RMSE, SMAPE, MASE, QuantileLoss, WinklerScore, MSIS, PIC')
    print(metrics[display_cols].to_string(index=False))
except FileNotFoundError:
    print('Metrics not generated yet. Run: python run_pipeline.py')

In [ ]:
try:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, series in zip(axes, ['gas', 'electricity', 'carbon']):
        for mode, marker, ls in [('core', 'o', '-'), ('macro', 's', '--')]:
            sub = metrics[(metrics['series_name']==series) & (metrics['mode']==mode)].copy()
            if sub.empty: continue
            sub = sub.sort_values('MAE')
            bars = ax.barh(
                sub['model'] + f'\n[{mode}]',
                sub['MAE'],
                color=PALETTE[mode], alpha=0.75, edgecolor='white', linewidth=0.5
            )
        ax.set_title(f'{series.capitalize()} — MAE (2017 eval)', fontsize=11, fontweight='bold')
        ax.set_xlabel('MAE')
        ax.axvline(0, color='gray', linewidth=0.7)
    fig.suptitle('Model MAE Comparison — Core vs Macro  [2017 test set]',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Plot error: {e}')

## 4 — 2018 Forecast vs Actual (per series)

In [ ]:
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('white')

for ax, series in zip(axes, ['gas', 'electricity', 'carbon']):
    ax.set_facecolor('white')
    for mode, ls, col_m in [('core', '-', PALETTE['core']), ('macro', '--', PALETTE['macro'])]:
        path = f'../outputs/forecasts/{series}_growth_pct_forecasts_all.csv'
        try:
            df = pd.read_csv(path, parse_dates=['date'])
            sub = df[df['mode'] == mode].sort_values('date')
            # Pick the best-ranked model (lowest MAE)
            best_model = None
            try:
                best_model = metrics[
                    (metrics['series_name']==series) & (metrics['mode']==mode)
                ].sort_values('MAE').iloc[0]['model']
                sub = sub[sub['model'] == best_model]
            except Exception:
                pass
            if sub.empty: continue
            label = f"{mode} ({best_model or 'best'})"
            ax.plot(sub['date'], sub['forecast'], color=col_m, linestyle=ls,
                    linewidth=2, marker='o', markersize=4, label=label)
            ax.fill_between(sub['date'], sub['lower_bound'], sub['upper_bound'],
                            alpha=0.12, color=col_m)
            if 'actual' in sub.columns and sub['actual'].notna().any():
                ax.plot(sub['date'], sub['actual'], color=PALETTE['actual'],
                        marker='D', markersize=6, linestyle='none',
                        label='Actual 2018', zorder=5)
        except FileNotFoundError:
            pass

    ax.axhline(0, color='#BDC3C7', linewidth=0.7)
    ax.set_title(f'{series.capitalize()} Growth (%) — 2018', fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(fontsize=8)
    ax.grid(True, color='#EAECEE', linewidth=0.7)

fig.suptitle('2018 Forecast vs Actual — Core and Macro Models',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5 — FES Monthly 2018 (Three Baselines)

In [ ]:
try:
    fes_monthly = pd.read_csv('../outputs/fes/fes_monthly_2018.csv', parse_dates=['date'])
    print(f'FES monthly data: {fes_monthly.shape}')
    display_cols = ['date','fes_core','fes_macro','fes_actual']
    present = [c for c in display_cols if c in fes_monthly.columns]
    print(fes_monthly[present].to_string(index=False))
except FileNotFoundError:
    print('FES not generated yet. Run: python run_pipeline.py')

In [ ]:
try:
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')

    for col, label, colour, ls in [
        ('fes_core',   'FES Core (core-only models)',      PALETTE['core'],   '-'),
        ('fes_macro',  'FES Macro (core + exogenous)',     PALETTE['macro'],  '--'),
        ('fes_actual', 'FES Actual (realised 2018)',       PALETTE['actual'], ':'),
    ]:
        if col in fes_monthly.columns:
            ax.plot(MONTHS, fes_monthly[col].values, color=colour,
                    linestyle=ls, linewidth=2.2, marker='o', markersize=5, label=label)

    ax.axhline(0, color='#95A5A6', linewidth=0.9)
    ax.set_title('UK Anticipatory Energy–Carbon Stress Index — Monthly 2018',
                 fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Month (2018)', fontsize=11)
    ax.set_ylabel('FES (sum of z-scores, training-period standardised)', fontsize=10)
    ax.legend(framealpha=0.92, fontsize=10)
    ax.grid(True, color='#EAECEE', linewidth=0.8)
    ax.set_xticks(range(12))
    ax.set_xticklabels(MONTHS)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Plot error: {e}')

## 6 — FES Component Z-Scores

In [ ]:
try:
    comp_table = pd.read_csv('../outputs/fes/fes_components_table.csv')
    print('\n── FES Component Table (z-score annual means, 2018) ──')
    print(comp_table.to_string(index=False))
except FileNotFoundError:
    print('FES components table not found. Run: python run_pipeline.py')

In [ ]:
try:
    # Component bar chart from fes_monthly
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.patch.set_facecolor('white')

    # Left: component z-scores annual mean
    ax = axes[0]
    ax.set_facecolor('white')
    components = ['gas_growth_pct', 'elec_growth_pct', 'carbon_growth_pct', 'Unc/RealVol']
    z_cols = {
        'FES_core':   ['z_gas_core',   'z_electricity_core',   'z_carbon_core',   'z_unc_core'],
        'FES_macro':  ['z_gas_macro',  'z_electricity_macro',  'z_carbon_macro',  'z_unc_macro'],
        'FES_actual': ['z_gas_actual', 'z_electricity_actual', 'z_carbon_actual', 'real_vol_actual'],
    }
    x = np.arange(4)
    width = 0.25
    colours_bar = {'FES_core': PALETTE['core'], 'FES_macro': PALETTE['macro'], 'FES_actual': PALETTE['actual']}
    for k, (label, cols) in enumerate(z_cols.items()):
        vals = [float(fes_monthly[c].mean()) if c in fes_monthly.columns else 0.0 for c in cols]
        bars = ax.bar(x + (k-1)*width, vals, width, label=label,
                      color=colours_bar[label], edgecolor='white', linewidth=0.6)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+(0.03 if v>=0 else -0.1),
                    f'{v:+.2f}', ha='center', fontsize=7.5, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Gas\nGrowth','Elec\nGrowth','Carbon\nGrowth','Unc/\nRealVol'], fontsize=9)
    ax.axhline(0, color='#95A5A6', linewidth=0.8)
    ax.set_title('FES Component Z-Scores (annual mean 2018)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Z-score'); ax.legend(fontsize=9)
    ax.grid(axis='y', color='#EAECEE', linewidth=0.8)

    # Right: FES total
    ax2 = axes[1]
    ax2.set_facecolor('white')
    fes_labels  = ['FES Core', 'FES Macro', 'FES Actual']
    fes_cols    = ['fes_core', 'fes_macro', 'fes_actual']
    fes_colours = [PALETTE['core'], PALETTE['macro'], PALETTE['actual']]
    fes_vals    = [float(fes_monthly[c].mean()) if c in fes_monthly.columns else 0.0
                   for c in fes_cols]
    bars = ax2.bar(fes_labels, fes_vals, color=fes_colours,
                   edgecolor='white', linewidth=0.8, width=0.5)
    for bar, v in zip(bars, fes_vals):
        ax2.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+(0.05 if v>=0 else -0.12),
                 f'{v:+.3f}', ha='center', fontsize=12, fontweight='bold')
    ax2.axhline(0, color='#95A5A6', linewidth=0.8)
    ax2.set_title('Total FES — Annual Mean 2018', fontsize=12, fontweight='bold')
    ax2.set_ylabel('FES (sum of z-scores)')
    ax2.grid(axis='y', color='#EAECEE', linewidth=0.8)

    fig.suptitle('Anticipatory Energy–Carbon Stress Index — Component Analysis 2018',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Plot error: {e}')

## 7 — Monthly Z-Score Heatmap

In [ ]:
try:
    import seaborn as sns

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, mode in zip(axes, ['core', 'macro', 'actual']):
        cols = [f'z_gas_{mode}', f'z_electricity_{mode}', f'z_carbon_{mode}']
        present = [c for c in cols if c in fes_monthly.columns]
        if not present: continue
        pivot = fes_monthly[present].copy()
        pivot.index = MONTHS
        pivot.columns = ['Gas', 'Electricity', 'Carbon']
        sns.heatmap(pivot, ax=ax, cmap='RdBu_r', center=0,
                    annot=True, fmt='.2f', linewidths=0.5,
                    cbar_kws={'label': 'z-score'})
        ax.set_title(f'FES {mode.capitalize()} — Monthly Z-Scores 2018',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('')

    plt.suptitle('Monthly Z-Score Heatmaps — Core | Macro | Actual',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Heatmap error: {e}')

## 8 — FES Methodology Summary

### Formula

$$
\text{FES}^{\text{core}}_t = z(\text{GasGrowth}_t) + z(\text{ElecGrowth}_t) + z(\text{CarbonGrowth}_t) + z(\text{Uncertainty}_t)
$$

$$
\text{FES}^{\text{macro}}_t = z(\hat{\text{Gas}}_t^\text{macro}) + z(\hat{\text{Elec}}_t^\text{macro}) + z(\hat{\text{Carbon}}_t^\text{macro}) + z(\text{Uncertainty}_t)
$$

$$
\text{FES}^{\text{actual}}_t = z(\text{GasGrowth}_t^\text{actual}) + z(\text{ElecGrowth}_t^\text{actual}) + z(\text{CarbonGrowth}_t^\text{actual}) + \text{RealVol}_t
$$

### Z-score standardisation

$$z(X_t) = \frac{X_t - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

where $\mu_{\text{train}}$, $\sigma_{\text{train}}$ come from **2005–2017** for all three variants — ensuring direct comparability.

### Important rules
- **Macro variables** (inflation, weather, GDP) are used **only as exogenous model inputs** — they do **not** enter the FES formula directly.
- **MAPE** is **excluded from model ranking** because growth rates near zero cause MAPE to explode. Ranking uses: MAE, RMSE, SMAPE, MASE, QuantileLoss, WinklerScore, MSIS, PIC.
- Uncertainty for FES_core/macro = **PI half-width** $(\text{UB} - \text{LB})/2$, z-scored using rolling historical volatility.
- Uncertainty for FES_actual = **cross-sectional std** of the three z-scored actual series per month.

In [ ]:
# Load and display all saved figures
from pathlib import Path
from IPython.display import Image, display as ipy_display

fig_dir = Path('../outputs/figures')
if fig_dir.exists():
    for png in sorted(fig_dir.glob('*.png')):
        print(f'\n── {png.name}')
        try:
            ipy_display(Image(filename=str(png), width=950))
        except Exception as e:
            print(f'  Could not display: {e}')
else:
    print('No figures found. Run: python run_pipeline.py')